# AbdomenNet Release Demo

This notebook provides a minimal public tutorial for:

1. running inference on the five release-demo CT cases
2. generating the public `pred_npz/test_epoch_0.npz` output
3. running a tiny smoke-training example to verify the training entry point

Before running the notebook, download the demo NIfTI cases from:

- https://drive.google.com/drive/folders/1uNKexIEE11j7Nf5LiFkiIGaTdCjW_nEm?usp=sharing

During peer review, this Google Drive folder is the default download source for the demo bundle.
After paper acceptance, the formal model release is planned to be mirrored on Hugging Face.

Place the downloaded files under `papers/revision_nc_20260329/release_demo/test_cases/`.


In [ ]:
from pathlib import Path
import os, json, textwrap, subprocess
import pandas as pd

repo_root = Path.cwd()
if not (repo_root / 'finetune').exists():
    raise RuntimeError('Please launch this notebook from the Acute_abdomen repository root.')

demo_root = repo_root / 'papers' / 'revision_nc_20260329' / 'release_demo'
test_cases_dir = demo_root / 'test_cases'

# Edit these two paths for your environment.
teacher_ckpt = Path('/absolute/path/to/teacher_checkpoint.pth')
finetune_ckpt = Path('/absolute/path/to/epoch_17.pth')
python_bin = Path('/absolute/path/to/python3.11')

print('repo_root =', repo_root)
print('demo_root =', demo_root)
print('test_cases_dir exists =', test_cases_dir.exists())
print('teacher_ckpt exists =', teacher_ckpt.exists())
print('finetune_ckpt exists =', finetune_ckpt.exists())
print('python_bin exists =', python_bin.exists())

assert teacher_ckpt.exists(), 'Please set teacher_ckpt to a valid checkpoint path.'
assert python_bin.exists(), 'Please set python_bin to a valid Python interpreter.'
# finetune_ckpt is required for the inference demo. The tiny smoke-training section below does not require it.


In [ ]:
required = [
    test_cases_dir / 'test_cases.csv',
    test_cases_dir / 'case1.nii.gz',
    test_cases_dir / 'case2.nii.gz',
    test_cases_dir / 'case3.nii.gz',
    test_cases_dir / 'case4.nii.gz',
    test_cases_dir / 'case5.nii.gz',
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError('Missing demo files:
' + '
'.join(missing))
print('All demo files are present.')


## Prepare a runnable inference CSV and config

The public dataset loader now resolves relative image paths against the CSV file location, but we still materialize an absolute-path CSV here for maximum portability and clarity.


In [ ]:
demo_csv = test_cases_dir / 'test_cases.csv'
df = pd.read_csv(demo_csv).copy()
df.iloc[:, 0] = [str((test_cases_dir / Path(p)).resolve()) if not Path(str(p)).is_absolute() else str(Path(p)) for p in df.iloc[:, 0]]
demo_csv_abs = demo_root / 'test_cases_abs.csv'
df.to_csv(demo_csv_abs, index=False)
print(demo_csv_abs)
display(df.head())

infer_cfg = demo_root / 'abdomennet_demo_infer.yaml'
infer_cfg.write_text(textwrap.dedent(f'''
data:
  train_dataset: {demo_csv_abs}
  val_dataset: {demo_csv_abs}
  test_dataset: {demo_csv_abs}
  batch_size: 1
  num_workers: 0
crops:
  global_crops_size: 518
model:
  n_last_blocks_list: [1, 4]
  num_classes: 11
  mean_slice_feature: false
  mst_type: concate
  backbone_update: false
  use_n_blocks: 4
  use_avgpool: true
  pretrained_weights: {teacher_ckpt}
  ckpt_path: {finetune_ckpt}
  num_decoder_layers: 3
  trans_nhead: 16
  trans_dim_feedforward_ratio: 4
optim:
  loss_type: BCE
  focal_loss_alpha: 0.25
  focal_loss_gamma: 2.0
  step_per_epoch: -1
  epochs: 30
  optimizer: sgd
  sgd_momentum: 0.9
  lr_scheduler: cosine
  weight_decay: 5e-4
  lr: 5e-4
  backbone_lr: 1e-3
  warmup_epochs: 0
  min_lr: 1.0e-06
  clip_grad: 3.0
  patch_embed_lr_mult: 0.2
  layerwise_decay: 1.0
  adamw_beta1: 0.9
  adamw_beta2: 0.999
trainer: EvalTrans25D_Trainer
''').strip() + '
', encoding='utf-8')
print(infer_cfg.read_text())


## Run inference demo

This command writes predictions to `pred_npz/test_epoch_0.npz`.


In [ ]:
infer_out = demo_root / 'demo_infer_output'
infer_out.mkdir(parents=True, exist_ok=True)
cmd = [
    str(python_bin),
    'finetune/run/run_trainer.py',
    '--config-file', str(infer_cfg),
    '--output-dir', str(infer_out),
]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print('returncode =', result.returncode)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
assert result.returncode == 0, 'Inference demo failed'


In [ ]:
pred_npz = infer_out / 'pred_npz' / 'test_epoch_0.npz'
assert pred_npz.exists(), pred_npz
print('prediction file:', pred_npz)
obj = __import__('numpy').load(pred_npz, allow_pickle=True)
print(obj.files)
payload = obj[obj.files[0]].item()
print('num cases:', len(payload))
print('sample keys:', list(payload.keys())[:3])


## Tiny smoke-training demo

This is **not** a real training recipe. It only verifies that the public training entry point launches and writes a checkpoint on the five demo cases.


In [ ]:
train_cfg = demo_root / 'abdomennet_demo_train_smoke.yaml'
train_cfg.write_text(textwrap.dedent(f'''
data:
  train_dataset: {demo_csv_abs}
  val_dataset: {demo_csv_abs}
  test_dataset: {demo_csv_abs}
  batch_size: 1
  num_workers: 0
crops:
  global_crops_size: 518
  global_crops_scale: [1.0, 1.0]
model:
  n_last_blocks_list: [1, 4]
  num_classes: 11
  mean_slice_feature: false
  mst_type: concate
  backbone_update: false
  use_n_blocks: 4
  use_avgpool: true
  pretrained_weights: {teacher_ckpt}
  num_decoder_layers: 3
  trans_nhead: 16
  trans_dim_feedforward_ratio: 4
optim:
  loss_type: BCE
  focal_loss_alpha: 0.25
  focal_loss_gamma: 2.0
  step_per_epoch: 0
  epochs: 1
  optimizer: sgd
  sgd_momentum: 0.9
  lr_scheduler: cosine
  weight_decay: 5e-4
  lr: 5e-4
  backbone_lr: 1e-3
  warmup_epochs: 0
  min_lr: 1.0e-06
  clip_grad: 3.0
  patch_embed_lr_mult: 0.2
  layerwise_decay: 1.0
  adamw_beta1: 0.9
  adamw_beta2: 0.999
trainer: FinetuneTrans25D_Trainer
''').strip() + '
', encoding='utf-8')
print(train_cfg.read_text())


In [ ]:
train_out = demo_root / 'demo_train_smoke_output'
train_out.mkdir(parents=True, exist_ok=True)
cmd = [
    str(python_bin),
    'finetune/run/run_trainer.py',
    '--config-file', str(train_cfg),
    '--output-dir', str(train_out),
]
print(' '.join(cmd))
result = subprocess.run(cmd, cwd=repo_root, capture_output=True, text=True)
print('returncode =', result.returncode)
print(result.stdout[-4000:])
print(result.stderr[-4000:])
assert result.returncode == 0, 'Smoke-training demo failed'


In [ ]:
ckpt_dir = train_out / 'checkpoints'
print('checkpoint dir exists =', ckpt_dir.exists())
if ckpt_dir.exists():
    print([p.name for p in ckpt_dir.glob('*.pth')])
